Para cada agente se debe identificar y explicar: Los estados, el entorno, la politica, recompensas y la funcion de accion. Para el proceso de exploracion y explotacion se debe aplicar una de las siguientes estrategias: Seleccion de acciones con intervalos de confianza, o Algoritmo del gradiente

Pregunta 2: CARRERA DE DADOS: Construye un agente que decida cuando avanzar o esperar en un juego de dados cencillo. Explica como aprende estrategias ganadoras

Estudiante: Delgado Ramos Jorge Luis

1. Primero indentificaremos los cinco elementos principales

1- El entorno: en esta parte definiremos las reglas logicas y las reglas fisicas del juego, esto tendra lo que es el dado de 6 caras, el oponente que tendra, y el limite de puntos para ganar por ejemplo 10 puntos o algo asi.

2- Los estados: podemos decir que es esa situacion actual que nuestro agente necesita ver para tomar una decicion, en esta ocacion definiremos al estado con 2 numeros Puntaje_Seguro_Agente, y Puntaje_Temporal_Turno.

3- La funcion de accion: Son las deciciones que el agente puede tomar en cualquier estado, en este caso como se nos indica en el enunciado solo seran 2: 0 esperar, y 1 avanzar. Donde podemos decir que si espera digamos que no arriesga a que algo pueda salir mal, sin embargo elegir entre avanzar puede llevar cierto riesgo.

4- Recompensas: esto sera el premio o castigo si el agente alcanza la meta tendra +1 pero si el oponente llega antes, entonces tendra -1, y podemos decir que tendra 0 si se realiza cualquier otra accion intermedia.

5- La politica: podemos decir que esta parte es el cerebro o la estrategia que tomara nuestro agente, y es generalmente una tabla de valores que elige de cada estado a la mejor accion, en este caso nuestro agente la usara para saber que hacer 

2. Para la explotacion y exploracion usaremos el metodo del algoritmo del gradiente para esto podemos decir que en lugar de aprender "cuántos puntos gano", el agente aprende "qué tanto me gusta esta acción" asignándole un valor H.

Para esto definiremos que si una accion le da mas recompensas entonces le subira la preferencia y a la otra le bajara.

In [ ]:
import random
import math

# --- PARAMETROS DEL JUEGO ---
META = 15
ALFA = 0.1  # Que tan rapido cambian las preferencias del agente

# --- MEMORIA DEL AGENTE ---
H = {}  # Diccionario de preferencias H(Estado, Accion)
recompensa_promedio = 0.0 
partidas_jugadas = 0

def inicializar_estado(estado):
    """Si el agente nunca ha visto este estado, crea sus preferencias en 0"""
    if estado not in H:
        H[estado] = {0: 0.0, 1: 0.0}  # 0: Esperar, 1: Avanzar

def probabilidades_softmax(estado):
    """Convierte las preferencias H en probabilidades % usando Softmax"""
    inicializar_estado(estado)
    h_esperar = H[estado][0]
    h_avanzar = H[estado][1]
    
    # Matematicas Softmax 
    exp_esperar = math.exp(h_esperar)
    exp_avanzar = math.exp(h_avanzar)
    suma_total = exp_esperar + exp_avanzar
    
    prob_esperar = exp_esperar / suma_total
    prob_avanzar = exp_avanzar / suma_total
    
    return prob_esperar, prob_avanzar

def elegir_accion_gradiente(estado):
    """Toma una decision basandose en las probabilidades calculadas"""
    prob_esperar, prob_avanzar = probabilidades_softmax(estado)
    
    # Exploracion implicita: Elegimos al azar pero sesgados por las probabilidades
    if random.random() < prob_esperar:
        return 0  # Esperar
    else:
        return 1  # Avanzar

# --- BUCLE DE APRENDIZAJE ---
for partida in range(5000):
    puntaje_agente = 0
    puntaje_oponente = 0
    
    # Mientras nadie llegue a la meta
    while puntaje_agente < META and puntaje_oponente < META:
        
        # --- TURNO DEL AGENTE ---
        puntos_turno = 0
        turno_activo = True
        
        while turno_activo:
            estado = (puntaje_agente, puntos_turno)
            prob_esperar, prob_avanzar = probabilidades_softmax(estado)
            
            accion = elegir_accion_gradiente(estado)
            
            # Ejecutar la accion
            if accion == 1:  # AVANZAR
                dado = random.randint(1, 6)
                if dado == 1:
                    puntos_turno = 0  # Pierde los puntos
                    turno_activo = False
                else:
                    puntos_turno += dado
                    if puntaje_agente + puntos_turno >= META:
                        turno_activo = False # Gano
            else:  # ESPERAR
                turno_activo = False
                
            # Fin de turno, procesar recompensa
            recompensa = 0
            if not turno_activo and (accion == 0 or (accion == 1 and dado != 1)):
                puntaje_agente += puntos_turno
                
            if puntaje_agente >= META:
                recompensa = 1 # GANO
                
            # --- ACTUALIZACION DEL GRADIENTE ---
            # Formula: H_nueva = H_vieja + Alfa * (Recompensa - Promedio) * (1 - Probabilidad)
            if accion == 1: # Si avanzo
                H[estado][1] += ALFA * (recompensa - recompensa_promedio) * (1 - prob_avanzar)
                H[estado][0] -= ALFA * (recompensa - recompensa_promedio) * prob_esperar
            else: # Si espero
                H[estado][0] += ALFA * (recompensa - recompensa_promedio) * (1 - prob_esperar)
                H[estado][1] -= ALFA * (recompensa - recompensa_promedio) * prob_avanzar

        # --- TURNO OPONENTE ---
        if puntaje_agente < META:
            dado_op = random.randint(1, 6)
            if dado_op != 1:
                puntaje_oponente += dado_op
                if puntaje_oponente >= META:
                    recompensa = -1 # EL AGENTE PIERDE
                    # Castigamos la ultima accion que causo la derrota
                    if accion == 1:
                        H[estado][1] += ALFA * (recompensa - recompensa_promedio) * (1 - prob_avanzar)
                    else:
                        H[estado][0] += ALFA * (recompensa - recompensa_promedio) * (1 - prob_esperar)

    # Actualizar la recompensa promedio de todas las partidas jugadas
    partidas_jugadas += 1
    recompensa_promedio += (recompensa - recompensa_promedio) / partidas_jugadas

Bueno ahora explicare como es que aprende las estrategias ganadoras el agente:

primero nuestro agente es ignorante pues las preferencias para H son todos los estados son 0, debido al Softmax, la probabilidad de Esperar o Avanzar es exactamente 50/50. Si el agente hace una jugada al azar y pierde, recibe -1. Como -1 es menor que su recompensa promedio, el algoritmo del gradiente le resta puntos de preferencia H a la acción que lo hizo perder. A medida que juega, si el agente descubre que "Esperar" cuando tiene 12 puntos seguros y 3 en el turno le da siempre la victoria (+1), la preferencia H de "Esperar" subirá tanto que el Softmax la convertirá en un 99% de probabilidad